# Konzept & Spezifikation: Agent Repair Agent (Terminal-basiert)

Dieses Dokument beschreibt die Architektur, den Ablauf und die Anforderungen für die Entwicklung des Agent Repair Agent. Der Agent dient dazu, Fehler im Projektcode oder im Verhalten von LLM-Agenten gezielt zu lokalisieren, zu analysieren und im Projektordner zu beheben.

In [5]:
import sys
from pathlib import Path
from typing import Any, Dict, Set, List

# 1. Projekt-Root-Verzeichnis für Jupyter Notebooks sicher ermitteln
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 2. Direkter Import der Logger-Klasse (umgeht das __file__-Problem im Notebook)
from src.agent.agent_universal_skript.class_logger_audit_execution import class_logger_audit_execution

# 3. Logger initialisieren (erkennt automatisch den Notebook-Namen und loggt sauber)
logger = class_logger_audit_execution.initialize_logger()

print(f"[*] Projekt-Root & Logger aktiv. Log-Datei: {class_logger_audit_execution.get_target_log_path()}")

In [ ]:
from pathlib import Path
from typing import Any, Dict, Set, List

# Globalen 

class Class_Agent_Repair_Agent:
    """Zentraler Agent Repair Agent zur automatisierten, kontextbezogenen
    Fehlererkennung und Code-Reparatur im Projekt.
    """
    # GLOBALE KONFIGURATIONEN MIT FALLBACK (_cfg.setdefault)
    _cfg: Dict[str, Any] = globals().setdefault("_AGENT_CONFIG", {})

    # Arbeits- und Ablagebereich für aktive Agenten-Eingriffe (Schreibrechte)
    DIR_AGENT_WORK: str = _cfg.setdefault("DIR_AGENT_WORK", "src/agent")

    # Auszuschließende Ordner und Pfade
    DIR_AGENT_EXCLUDE_KNOWLEDGE: Set[str]           = _cfg.setdefault("DIR_AGENT_EXCLUDE_KNOWLEDGE",{"privacy", "text", "temp", "scratch", ".git", "__pycache__", ".venv", "venv", ".ipynb_checkpoints"})
    DIR_AGENT_INCLUDE_KNOWLEDGE: bool               = _cfg.setdefault("DIR_AGENT_INCLUDE_KNOWLEDGE", True)
    FORMATE_INCLUDE_READABLE_EXTENSIONS: Set[str]   = _cfg.setdefault("FORMATE_INCLUDE_READABLE_EXTENSIONS",{".py",".ipynb",".json",".log",".txt",".md",".jsonl",".yaml",".yml",".ini",".cfg",},)
    FORMATE_WRITABLE_FILE_EXTENSIONS: Set[str]      = _cfg.setdefault("FORMATE_WRITABLE_FILE_EXTENSIONS", {".py", ".ipynb", ".md", ".txt"})
    FORMATE_EXCLUDE_EXTENSIONS: Set[str]            = _cfg.setdefault("FORMATE_EXCLUDE_EXTENSIONS",{".pyc", ".pyo", ".pyd", ".dll", ".exe", ".so", ".bin", ".zip"},)
    OLAMA_MODEL_CODE_LOW    = _cfg.setdefault("OLAMA_MODEL_CODE",   "codestral:22b")
    OLAMA_MODEL_CODE_HIGH   = _cfg.setdefault("OLAMA_MODEL_CODE",   "qwen3-coder-next-ud-q4-k-xl")
    OLAMA_MODEL_LOGIC_LOW   = _cfg.setdefault("OLAMA_MODEL_LOGIC",  "deepseek-r1:8b")
    OLAMA_MODEL_LOGIC_HIGH  = _cfg.setdefault("OLAMA_MODEL_LOGIC",  "qwen3.5:122b")
    OLAMA_MODEL_TEXT_LOW    = _cfg.setdefault("OLAMA_MODEL_LOGIC",  "moondream")
    OLAMA_MODEL_TEXT_HIGH   = _cfg.setdefault("OLAMA_MODEL_LOGIC",  "gemma3:4b")

    def __init__(self):
        """Initialisiert den Repair Agent, bindet den zentralen Logger ein
        und prüft die Startumgebung.
        """
        try:
            self.logger = class_logger_audit_execution
            self.logger.info("Class_Agent_Repair_Agent erfolgreich instanziiert.", module="RepairAgent",)
        except NameError:
            print("[WARNUNG] class_logger_audit_execution im Namespace nicht gefunden.")
            self.logger = None

    @classmethod
    def get_project_root(cls) -> Path:
        """Ermittelt die Projekt-Root über den Logger oder Fallback."""
        try:
            return class_logger_audit_execution._detect_project_root()
        except Exception:
            return (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())

    @classmethod
    def get_work_path(cls) -> Path:
        """Gibt den absoluten Pfad zum erlaubten Agenten-Arbeitsordner zurück."""
        root = cls.get_project_root()
        target = root / cls.DIR_AGENT_WORK
        target.mkdir(parents=True, exist_ok=True)
        return target

    @classmethod
    def get_effective_knowledge_paths(cls) -> List[Path]:
        """Ermittelt die finale Wissensbasis per Mengenabzug (Include minus Exclude)
        unter Berücksichtigung der Verzeichnis- und Format-Guardrails.
        """
        root = cls.get_project_root()
        if not cls.DIR_AGENT_INCLUDE_KNOWLEDGE:
            return []

        effective_paths: List[Path] = []

        for path in root.rglob("*"):
            # Verzeichnis-Ausschlüsse prüfen
            if not set(path.parts).isdisjoint(cls.DIR_AGENT_EXCLUDE_KNOWLEDGE):
                continue

            # Dateispezifische Format-Filterung prüfen
            if path.is_file():
                ext = path.suffix.lower()
                
                # Format-Excludes prüfen
                if ext in cls.FORMATE_EXCLUDE_EXTENSIONS:
                    continue
                
                # Format-Includes prüfen
                if cls.FORMATE_INCLUDE_READABLE_EXTENSIONS and ext not in cls.FORMATE_INCLUDE_READABLE_EXTENSIONS:
                    continue

                effective_paths.append(path)

        return effective_paths

## 1. Ziel & Zweck des Vorhabens

Das Hauptziel dieses Systems ist die automatisierte Selbstreparatur und Korrektur von Codebase-Beständen basierend auf natürlicher Benutzereingabe.

Wenn ein LLM-Agent während des Projektverlaufs unerwünschte Ergebnisse liefert, den Kontext falsch versteht oder fehlerhaften Code generiert hat, übernimmt der Agent Repair Agent die Korrektur. Er agiert als spezialisiertes Werkzeug zur Qualitäts- und Funktionssicherung.

## 2. Abfolge & Funktionsweise

Der Reparaturprozess gliedert sich in folgende Schritte:

A. Erfassung & Verstehen der Benutzereingabe

Der Benutzer gibt Feedback über das Terminal oder ein Notizbuch ein (z. B. "Die letzte Aufgabe hat kein sauberes Ergebnis geliefert", "Ich meinte eigentlich A, B, C und D", oder "Funktion X wirft einen Fehler auf").

Das System analysiert die Eingabe und erkennt automatisch, dass es sich um eine Korrektur-/Reparaturanforderung für das Projekt handelt.

B. Fehler-Diagnose & Zuordnung

Der Agent prüft, welcher Teil des Projekts oder welcher Sub-Agent für den Fehler verantwortlich ist.

Er kategorisiert das Problem (z. B. Logikfehler im Prompt, falscher Parameteraufruf, defekte Pipeline, Syntax- oder Runtime-Error).

C. Präzise Code-Lokalisierung

Der Agent durchsucht den gesamten Projektordner nach den relevanten Dateien, Funktionen oder Codeblöcken.

Er identifiziert exakt die Stellen, die angepasst werden müssen, ohne unbeteiligten Code zu beeinträchtigen.

D. Subtraktive & Additive Code-Regulierung (Clean Patching)

Kein Datenverlust: Es wird verhindert, dass funktionierender Code ungewollt gelöscht oder überschrieben wird.

Keine Überlagerung: Alter, obsoleter Code wird sauber entfernt oder ersetzt, anstatt doppelten/redundanten Code aufzubauen.

Schnittstellen-Erhalt: Alle bestehenden Abhängigkeiten, Funktionsaufrufe und Schnittstellen im Projekt bleiben intakt und kompatibel.

## 3. Betriebsmodus: Terminal-Schnittstelle (CLI / Shell)

Um maximale Unabhängigkeit von grafischen Benutzeroberflächen (GUIs) oder speziellen IDE-Setups zu gewährleisten, wird der Agent primär für die Terminal-Oberfläche entwickelt:

CLI-Steuerung: Aufruf direkt über die Kommandozeile (z. B. python -m repair_agent "Fehlermeldung / Anweisung").

Autonome Ordner-Navigation: Der Agent arbeitet direkt im Ziel-Projektverzeichnis, liest Dateistrukturen aus und wendet Änderungen lokal an.

Jupyter Notebook Integration: Ausführung und Tests können aus einem Steuerungs-Notebook heraus getriggert werden, wobei der zugrundeliegende Agent im Terminal-Modus arbeitet.

## 4. Anwendungsbeispiele (Use Cases)SzenarioBenutzereingabeAktion des Repair AgentsErgebnisabweichung"Du hast mir bei der letzten Aufgabe kein sauberes Ergebnis gegeben."Analysiert letzte Ausgabe/Logfiles, identifiziert fehlerhaften Formatter im Code und korrigiert die Ausgabe-Logik.Klarstellung / Logikfehler"Nein, ich meinte A, B, C und D statt X."Lokalisiert das betroffene Modul, passt Parameter und Kontrollfluss an, ohne bestehende Importe/Verbindungen zu unterbrechen.Laufzeitfehler"IndexError in script.py Zeile 42."Öffnet script.py, behebt den Boundary-Check und stellt sicher, dass Tests/Schnittstellen weiterhin grün sind.

## 5. Zusammenfassung der Kernanforderungen

Kontextverständnis: Automatische Inferenz von Korrekturwünschen aus natürlicher Sprache.

Präzisions-Patching: Punktgenaue Code-Modifikation (weder Overwriting noch Zombie-Code).

Ordner-Awareness: Vollständiger Zugriff auf und Verständnis der Dateistruktur des Projektordners.

UI-Unabhängigkeit: Vollfunktionsfähig auf der Shell/Terminal-Ebene.